# NLP Practical Exam — Text Processing + Language Modeling (90 minutes)

**Instructions**
- Work in this notebook only.
- Write short, clear comments to justify *tool choices* (regex vs NLTK, etc.).
- Do **not** use external NLP libraries beyond **NLTK**, **NumPy**, **PyTorch** (PyTorch not needed here).
- Keep outputs readable (print key variables).

**Total: 10 points**


## Given text

```python
text = ("In mid-February 2026, the CEO of OpenAI, Sam Altman, visited Barcelona. He is 1.86m tall and met with researchers from U.P.C. and U.N.E.S.C.O. A report valued the project at $3.2 billion.")
```

> Treat the text as *synthetic exam data* (no fact-checking needed).


## Questions

1. **(1 pt)** Sentence splitting using **regex + NLTK**.
2. **(1 pt)** Regex normalization: acronyms, height meters→centimeters, money `$X.Y billion` → `x point y billion` (words).
3. **(1 pt)** Lowercase **except** proper nouns; join multiword proper nouns with underscore (e.g., `Sam Altman → Sam_Altman`). Keep acronyms uppercase.
4. **(1 pt)** Tokenize (tool of your choice).
5. **(1 pt)** Remove stopwords (tool of your choice); keep entity tokens.
6. **(1 pt)** Create bigrams with pure Python.
7. **(2 pt)** Build a bigram LM (MLE) and `predict_next(prev_word, top_k=3)`.

8. **(2 pt)** Implement a simple **BPE** on: `corpus = "low lower newest widest"` (≥5 merges or until no merges).
9. **(1 pt)** Compute Accuracy/Precision/Recall/F1 for an invented confusion matrix (explain with comments).


In [27]:
import re
import math
import nltk
from collections import Counter, defaultdict

# NLTK downloads (safe to run multiple times)
nltk.download("punkt", quiet=True)
nltk.download("stopwords", quiet=True)

from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords

text = ("In mid-February 2026, the CEO of OpenAI, Sam Altman, visited Barcelona. "
        "He is 1.86m tall and met with researchers from U.P.C. and U.N.E.S.C.O. "
        "A report valued the project at $3.2 billion.")

print(text)


In mid-February 2026, the CEO of OpenAI, Sam Altman, visited Barcelona. He is 1.86m tall and met with researchers from U.P.C. and U.N.E.S.C.O. A report valued the project at $3.2 billion.


## Q1

In [28]:
# Q1 (1 pt): Sentence splitting (regex + NLTK)
# - Use regex to protect acronyms like U.P.C. so they don't break sentence boundaries.
# - Then use nltk.sent_tokenize.
#
# Return: sentences (list of strings)

# TODO: implement protect_acronym_dots and restore_acronym_dots (or equivalent)
def protect_acronym_dots(text): # This is a implementation that basically what they do is replace the dots in acronyms.
    return re.sub(r'\b([A-Z])\.(?=[A-Z]\.)', r'\1<dot>', text) # This regex search for capital letter that are followed by a dot and other capital letter
def restore_acronym_dots(text): # This is a fuction that restore the diferents acronyms that we protected in the previous fuction that we already did.
    return re.sub(r'<dot>', r'.', text) # And that is the regex for serch <dot> and replace it with a real dot.
# TODO: apply sent_tokenize
protected_text = protect_acronym_dots(text) # For the first step i protect the acronyms.
sentences = sent_tokenize(protected_text) # Then i apply the set_tokenize to the text that we already protect.
sentences = [restore_acronym_dots(s) for s in sentences] # For the last step i restore the acronyms in the sentences of the results.

print(sentences) # This basically print the sentences that we get after applying the sentence splitting.


['In mid-February 2026, the CEO of OpenAI, Sam Altman, visited Barcelona.', 'He is 1.86m tall and met with researchers from U.P.C.', 'and U.N.E.S.C.O.', 'A report valued the project at $3.2 billion.']


After seeing the output, I realized that the last period of acronyms like U.N.E.S.C.O. was not protected and therefore is causing the sentence to be broken incorrectly. I suppose it has to do with the REGEX.

In [ ]:
git add .
git commit -m "I completed Q3, although it was a little more complicated to do simply because I had to stop and think about how I was going to solve it and what steps I could divide it into."
git push

SyntaxError: invalid syntax (2134615836.py, line 1)

## Q2

In [30]:
# Q2 (1 pt): Regex normalization
# Convert:
#  - U.P.C. -> UPC, U.N.E.S.C.O. -> UNESCO (general rule: remove dots in acronyms)
#  - 1.86m -> 186 centimeters (general: X.YZm -> int(round(float(X.YZ)*100)) centimeters)
#  - $3.2 billion -> three point two billion  (digits 0-9 are enough)
#
# Return: text_norm
# Teniendo en cuenta todo lo que se pide en el enunciado escrito en los #, aqui abajo empiezo el código:

text_norm = text

# 1) Remove dots in acronyms (general rule)
text_norm = re.sub(r'\b(?:[A-Z]\.)+[A-Z]\.?',lambda m: m.group(0).replace('.', ''),text_norm)

# 2) Convert 1.86m -> 186 centimeters
text_norm = re.sub(r'(\d+\.\d+)m',lambda m: f"{int(round(float(m.group(1)) * 100))} centimeters",text_norm)

# 3) Convert $3.2 billion -> three point two billion (digits 0-9 are enough)
digit_words = {
    '0': 'zero', '1': 'one', '2': 'two', '3': 'three',
    '4': 'four', '5': 'five', '6': 'six',
    '7': 'seven', '8': 'eight', '9': 'nine'
}

def convert_billion(match):
    number = match.group(1)
    integer, decimal = number.split('.')
    return (
        " ".join(digit_words[d] for d in integer)
        + " point "
        + " ".join(digit_words[d] for d in decimal)
        + " billion"
    )

text_norm = re.sub(r'\$(\d+\.\d+)\s*billion',convert_billion,text_norm)

print(text_norm)



In mid-February 2026, the CEO of OpenAI, Sam Altman, visited Barcelona. He is 186 centimeters tall and met with researchers from UPC and UNESCO A report valued the project at three point two billion.


For the first case, I did exactly what it asked: remove the periods from the various acronyms using a regular expression. This expression detects sequences like U.P.C. and simply deletes the periods.

For the second case, I basically had to convert measurements like 1.86m by multiplying the number by 100 and then converting it to centimeters—nothing complicated.

For the third exercise, I transformed quantities like 3.2 billion by converting each digit into its corresponding word, always keeping the word "billion."

## Q3

In [ ]:
# Q3 (1 pt): Lowercase except proper nouns + underscore multiword proper nouns
# Requirements:
# - Convert to lowercase except:
#   - Acronyms (ALL CAPS) stay uppercase (e.g., UNESCO, UPC, CEO)
#   - MixedCase tokens stay as-is (e.g., OpenAI)
#   - Multiword proper nouns joined with underscore (Sam Altman -> Sam_Altman) and preserved
#
# Return: text_case
import re
text_case = text

# First i would unit the own names of a some words, like Sam Altam in Sam_Altman
# I do a list of multiword to propose nouns that we already know about the text
multiword_proper_nouns = ["Sam Altman"]

for noun in multiword_proper_nouns:
    text_case = text_case.replace(noun, noun.replace(" ", "_"))

# Now i am going to convert all to lowercase except the acronymus in uppercase (The patrin would be two or more letter in uppercase followed)
# and the other case would be the MixedCase toekens, that is like, at least one capital letter followed by lowercase letters.
def preserve_case(match):
    token = match.group(0)
    # Acrónimos: todo mayúsculas (2+ letras)
    if re.fullmatch(r'[A-Z]{2,}', token):
        return token
    # MixedCase: una mayúscula seguida de minúsculas
    elif re.fullmatch(r'[A-Z][a-z]+', token):
        return token
    else:
        return token.lower()
    
# 3) Now we do the final step in wich we aplicated the regex in each word.
text_case = re.sub(r'\b\w+\b', preserve_case, text_case)
print(text_case)


In mid-February 2026, the CEO of openai, sam_altman, visited Barcelona. He is 1.86m tall and met with researchers from u.p.c. and u.n.e.s.c.o. a report valued the project at $3.2 billion.


To complete the task, I did the following:
- First, I joined proper nouns with underscores "_", as in the example of Sam Altman in Sam_Altman.
- Then, I converted all the text to lowercase, except for acronyms, which remained in uppercase, and mixed-case tokens like OpenIA.
- Finally, I applied this rule word by word using a regular expression to ensure that each case was respected.

## Q4

In [ ]:
# Q4 (1 pt): Tokenization
# Use a tokenizer of your choice (e.g., nltk.word_tokenize).
# Return: tokens (list)

tokens = None

# print(tokens)


## Q5

In [ ]:
# Q5 (1 pt): Stopword removal
# - Remove English stopwords
# - Do NOT remove entity tokens like OpenAI, Sam_Altman, Barcelona, UNESCO, UPC
# Return: tokens_nostop

tokens_nostop = None

# print(tokens_nostop)


## Q6

In [ ]:
# Q6 (1 pt): Bigrams with pure Python (no NLTK bigrams helper)
# Return: bigrams = [(w1, w2), ...]

bigrams = None

# print(bigrams)


## Q7

In [ ]:
# Q7 (2 pt): Bigram Language Model + next-word prediction
# Build:
# - bigram_counts[(w1,w2)]
# - context_counts[w1]
# - model[w1][w2] = P(w2|w1) = count(w1,w2)/count(w1)
#
# Then implement:
# def predict_next(prev_word, model, top_k=3): -> list[(next_word, prob)] sorted

bigram_counts = None
context_counts = None
model = None

def predict_next(prev_word, model, top_k=3):
    # TODO
    return None

# Example:
# print(predict_next("OpenAI", model, top_k=3))


## Q8

In [ ]:
# Q8 (2 pt): Simple BPE (Byte Pair Encoding) on a tiny corpus
corpus = "low lower newest widest"

# Requirements:
# - Represent each word as characters + </w>
# - Compute pair frequencies (weighted by word frequency)
# - Merge most frequent pair
# - Do at least 5 merges (or stop if no pairs)
#
# Deliver:
# - merges: list of merges in order
# - final segmented version of each word

merges = None

# TODO: implement BPE helper functions:
# - get_vocab_from_corpus
# - get_pair_frequencies
# - merge_pair_in_vocab

# print(merges)


## Q9

In [32]:
# Q9 (1 pt): Metrics — Accuracy, Precision, Recall, F1
# Invent a confusion matrix (TP, FP, FN, TN) and compute metrics.
# Explain each formula briefly in comments.

TP = 60
FP = 10
FN = 15
TN = 50

accuracy = (TP + TN) / (TP + TN + FP + FN)
precision = TP / (TP + FP)
recall = TP / (TP + FN)
f1 = 2 * precision * recall / (precision + recall) # balance between both

print(f"Accuracy: {accuracy:.2f}")
print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")
print(f"F1: {f1:.2f}")

Accuracy: 0.81
Precision: 0.86
Recall: 0.80
F1: 0.83


- TP -> True positives are the cases that are correctly predicted as positive.
- FP -> False positives, in this case, will be the cases predicted as positive but which in the end were negative in reality.
- FN -> False negatives, now the opposite case to the previous one, these will be cases predicted as negative but which were actually positive.
- TN -> True negatives are the cases that were correctly predicted as negative.

Accuracy is the percentage of correct predictions. Precision, on the other hand, is the proportion of positive predictions that are correct. Recall is the proportion of actual positives detected. And last but not least, F1 represents the balance between precision and recall.

I found this exercise quite easy because we had already done the exact same exercise in class, we understood it, and the teacher explained it, so I had no difficulty doing it.